# RAG(문서 검색) 방식 시연 — 임베딩 + 유사도 검색 + PydanticAI Agent 답변

RAG(Retrieval-Augmented Generation)는 **문서를 벡터로 변환(임베딩)하고, 질문과 유사한 문서를 검색하여 LLM에게 제공**하는 구조입니다.
LLM이 직접 지식을 기억하는 대신, 검색된 문서를 근거로 답변하므로 환각(Hallucination)을 줄일 수 있습니다.

**파이프라인**: 문서 로드 → 청킹 → 임베딩 → 유사도 검색 → LLM 답변 생성

- **[장점]** 비정형 지식 Q&A / 근거 기반 답변 / 문서 추가만으로 지식 확장 / 환각 감소
- **[단점]** 실시간 데이터 불가 / 검색 품질이 답변 품질 결정 / 다단계 추론에 약함

## 환경 설정

필요한 라이브러리를 임포트하고 환경변수를 로드합니다.
- `numpy`: 벡터 연산 (코사인 유사도 계산)
- `OpenAI`: 임베딩 API 호출용 클라이언트
- `PydanticAI Agent`: 검색 결과를 바탕으로 답변을 생성하는 LLM Agent

In [ ]:
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from pydantic_ai import Agent

# .env 파일에서 OPENAI_API_KEY 로드
load_dotenv()

# 임베딩 API 호출용 OpenAI 클라이언트
openai_client = OpenAI()
# 검색 대상 문서가 있는 디렉토리
DOCS_DIR = Path("sample_docs")

## 1. 문서 로드 + 청킹

긴 문서를 통째로 임베딩하면 의미가 희석되므로, **단락(빈 줄) 기준으로 작은 청크로 분할**합니다.
각 청크에 출처(filename)를 함께 저장하여 나중에 "어떤 문서에서 왔는지" 추적할 수 있게 합니다.

- `max_chunk_size=500`: 청크 최대 길이 (너무 크면 검색 정확도 저하, 너무 작으면 문맥 손실)

In [ ]:
def chunk_documents(max_chunk_size: int = 500) -> list[dict]:
    """문서를 단락 기준으로 청크 분할하고, 출처 정보를 함께 저장한다."""
    chunks = []
    for md_file in sorted(DOCS_DIR.glob("*.md")):
        content = md_file.read_text(encoding="utf-8")
        current_chunk = ""
        # 빈 줄(\n\n) 기준으로 단락을 분리
        for para in content.split("\n\n"):
            para = para.strip()
            if not para:
                continue
            # max_chunk_size 이하이면 현재 청크에 병합
            if len(current_chunk) + len(para) < max_chunk_size:
                current_chunk += ("\n\n" + para) if current_chunk else para
            else:
                # 크기 초과 시 현재 청크를 저장하고 새 청크 시작
                if current_chunk:
                    chunks.append({"text": current_chunk, "source": md_file.name})
                current_chunk = para
        # 마지막 청크 저장
        if current_chunk:
            chunks.append({"text": current_chunk, "source": md_file.name})
    return chunks

## 2. 임베딩

텍스트를 **고차원 벡터(숫자 배열)**로 변환합니다.
의미가 비슷한 텍스트는 벡터 공간에서 가까이 위치하게 되어, 유사도 계산이 가능해집니다.

- `text-embedding-3-small`: OpenAI의 경량 임베딩 모델 (1536차원, 저비용)

In [ ]:
def embed_texts(texts: list[str]) -> np.ndarray:
    """텍스트 리스트를 임베딩 벡터 배열로 변환한다."""
    # OpenAI Embedding API 호출 — 여러 텍스트를 한번에 배치 임베딩
    response = openai_client.embeddings.create(model="text-embedding-3-small", input=texts)
    # 각 item의 embedding 벡터를 numpy 배열로 변환
    return np.array([item.embedding for item in response.data])

## 3. 검색 (Retrieval)

질문과 각 청크 간의 **코사인 유사도**를 계산하여, 가장 관련 있는 상위 `top_k`개 청크를 반환합니다.

- 코사인 유사도 = 정규화 벡터의 내적 (값이 1에 가까울수록 의미가 유사)
- 벡터를 L2 정규화한 뒤 내적(dot product)하면 코사인 유사도와 동일

In [ ]:
def retrieve(query: str, chunks: list[dict], chunk_embeddings: np.ndarray, top_k: int = 3) -> list[dict]:
    """질문과 유사한 청크를 코사인 유사도로 검색하여 상위 top_k개를 반환한다."""
    # 질문을 임베딩 벡터로 변환
    query_emb = embed_texts([query])[0]
    # L2 정규화 — 코사인 유사도 계산 준비
    q_norm = query_emb / np.linalg.norm(query_emb)
    c_norm = chunk_embeddings / np.linalg.norm(chunk_embeddings, axis=1, keepdims=True)
    # 내적 = 코사인 유사도 (정규화된 벡터끼리)
    similarities = c_norm @ q_norm
    # 유사도 높은 순으로 정렬하여 상위 top_k개 인덱스 추출
    ranked = np.argsort(similarities)[::-1][:top_k]
    results = []
    print(f"\n[검색 결과] 질문: '{query}'")
    for rank, idx in enumerate(ranked):
        print(f"  #{rank+1} (유사도: {similarities[idx]:.4f}) [{chunks[idx]['source']}]")
        results.append({"text": chunks[idx]["text"], "source": chunks[idx]["source"], "score": float(similarities[idx])})
    return results

## 4. RAG 파이프라인

전체 흐름을 하나의 함수로 묶습니다:
1. 문서 청킹 → 2. 임베딩 → 3. 유사도 검색 → 4. 검색 결과를 Agent instructions에 주입하여 답변 생성

검색된 문서를 **시스템 프롬프트에 직접 주입**하고, "문서에 없는 내용은 답하지 마라"는 규칙을 추가하여 환각을 방지합니다.

> **참고**: `run_rag_pipeline`은 내부에서 `await agent.run()`을 호출하므로 `async def`로 정의합니다.

In [ ]:
# async def — 내부에서 await agent.run()을 사용하므로 비동기 함수로 정의
async def run_rag_pipeline(query: str) -> str:
    """RAG 파이프라인: 청킹 → 임베딩 → 검색 → Agent 답변 생성"""
    print(f"[RAG Pipeline] 질문: {query}")

    # 1단계: 문서 로드 + 청킹
    chunks = chunk_documents()
    print(f"[청킹] 총 {len(chunks)}개 청크")

    # 2단계: 모든 청크를 임베딩 벡터로 변환
    chunk_embeddings = embed_texts([c["text"] for c in chunks])
    print(f"[임베딩] 완료: {chunk_embeddings.shape}")

    # 3단계: 질문과 유사한 청크 top-3 검색
    retrieved = retrieve(query, chunks, chunk_embeddings, top_k=3)

    # 4단계: 검색 결과를 context 문자열로 구성
    context = "\n\n---\n\n".join(
        f"[문서 {i+1}: {c['source']}]\n{c['text']}" for i, c in enumerate(retrieved)
    )

    # PydanticAI Agent — instructions에 검색 결과 주입 + 환각 방지 규칙
    rag_agent = Agent(
        "openai:gpt-5.4",
        instructions=(
            "당신은 TechShop의 고객 상담 에이전트입니다.\n"
            "아래 제공된 문서를 바탕으로 고객의 질문에 답변하세요.\n\n"
            "규칙:\n"
            "1. 반드시 제공된 문서 내용만을 근거로 답변하세요.\n"
            "2. 문서에 없는 내용은 '해당 정보를 찾을 수 없습니다'라고 답하세요.\n"
            "3. 답변 끝에 참조한 문서명을 표기하세요.\n\n"
            f"[참조 문서]\n{context}"
        ),
    )

    print("\n[답변 생성 중...]")
    # await agent.run() — Jupyter 노트북에서는 run_sync() 대신 사용
    result = await rag_agent.run(query)
    print(f"\n[최종 답변]\n{result.output}")
    return result.output

## 실행: 환불 정책 질문

`sample_docs/` 폴더에 있는 환불 관련 문서를 검색하여 답변합니다.
어떤 청크가 검색되었는지, 유사도 점수는 얼마인지 확인해 보세요.

In [ ]:
# run_rag_pipeline은 async 함수이므로 await로 호출
await run_rag_pipeline("우리 회사 제품의 환불 정책이 어떻게 되나요?")

## 실행: 해외 배송 질문

같은 파이프라인으로 다른 질문을 실행합니다. 검색되는 청크가 달라지는 것을 확인하세요.
만약 문서에 해외 배송 정보가 없다면, Agent는 규칙에 따라 "정보를 찾을 수 없다"고 답해야 합니다.

In [ ]:
# 다른 주제의 질문으로 파이프라인 재실행
await run_rag_pipeline("해외 배송은 얼마나 걸리나요? 배송비는 어떻게 되나요?")